In [1]:
library(Seurat)
library(circlize)
library(RColorBrewer)
library(dplyr)
library(Cairo)
library(ComplexHeatmap)
library(magick)
library(gtools)
library(msigdbr)
library(readODS)

Le chargement a nécessité le package : SeuratObject

Le chargement a nécessité le package : sp


Attachement du package : ‘SeuratObject’


Les objets suivants sont masqués depuis ‘package:base’:

    intersect, t


circlize version 0.4.16
CRAN page: https://cran.r-project.org/package=circlize
Github page: https://github.com/jokergoo/circlize
Documentation: https://jokergoo.github.io/circlize_book/book/

If you use it in published research, please cite:
Gu, Z. circlize implements and enhances circular visualization
  in R. Bioinformatics 2014.

This message can be suppressed by:
  suppressPackageStartupMessages(library(circlize))



Attachement du package : ‘dplyr’


Les objets suivants sont masqués depuis ‘package:stats’:

    filter, lag


Les objets suivants sont masqués depuis ‘package:base’:

    intersect, setdiff, setequal, union


Le chargement a nécessité le package : grid

ComplexHeatmap version 2.18.0
Bioconductor page: http://bioconductor.org/packages/ComplexHeatmap/
Github 

In [2]:
load("/work/project/fragencode/workspace/umr_1141/results/epilepsie/spatial/seurat_obj/whole_clustering_subclustering_Seurat_objects.RData")

In [3]:
options(repr.plot.width = 30, repr.plot.height = 30)

In [4]:
seurat_object_whole_clustering@meta.data$condition_time <- paste0(seurat_object_whole_clustering@meta.data$condition, "_", seurat_object_whole_clustering@meta.data$time)

In [5]:
Idents(seurat_object_whole_clustering) <- "seurat_custom_clusters"

In [232]:
data <- read_ods("/home/adufour/work/table/epilepsie/additional_file5.ods")

In [270]:
df_merge <- data[data$p_val_adj < 0.05,]

In [234]:
df_merge_filtered <- df_merge[df_merge$avg_log2FC > 1 | df_merge$avg_log2FC < -1,]

In [271]:
df_merge_filtered <- df_merge[df_merge$avg_log2FC > 0 | df_merge$avg_log2FC < -1,]

In [272]:
go_terms <- list(antigen_pres = c("Ctse", "Fcgr1", "H2-M3", "Tapbp", "H2-Oa", "Tap2", "Cd74", "Pikfyve", "Traf6", "Clec4a2", "Ifi30", "H2-Ea", "H2-Aa", "Tap1", "H2-DMb2", "Ctss", "Mfsd6", "Fcer1g", "B2m", "H2-K1", "H2-T23", "H2-Ab1"),
                 chaperone_auto = c("Ctsa", "Gfap", "Snca", "Plk3", "Eef1a1"),
                 mg_activation = c("Il4", "Mmp8", "Trpv1", "Ager", "Ttbk1", "Ifngr1", "Il13", "Nampt", "Nr1d1", "Clu", "App", "Ifngr2", "Trem2", "Aif1", "Tnf", "Il33", "Casp1", "Snca", "Tlr2", "Stap1", "Ctsc", "Tyrobp", "Tlr3", "Cx3cl1", "Ldlr", "Atm", "Grn", "Lrrk2", "C1qa", "Tlr4", "Tlr1", "Tlr9", "C5ar1", "Tlr6", "Fpr2", "Cx3cr1", "Jun", "Ifng", "Tafa3", "Sphk1", "Mir155", "Mir128-2", "Mir124a-3", "Mir181c", "Mir128-1", "Mir124a-1", "Cst7", "Syt11", "Mir223", "Mir124a-2", "Mir142"),
                 synapse_prun = c("Trem2", "C3", "Epha4", "Ngef", "Cdk5", "C1qa", "C1qc", "C1qb"),
                long_terms = c("Ccnd2", "Apoe", "Eif2ak4", "Ctns", "Ehmt2", "Srf", "Slc2a4", "Sgk1", "Adcy1", "Rps6kb1", "Gria1", "Kat2a", "Glud1", "Adcy8", "Arc", "Nfatc4", "Pja2", "Grin1", "Snap25", "Calb1", "Mtor", "Prkcz", "Tacr1", "Mecp2", "Ldlr", "Drd2", "Camk4", "Egr1", "Shank1", "Cpeb3", "Ptchd1", "Reln", "Lrrn4", "Npas4", "Rgs14", "Chd8", "Ntrk2", "Tac1", "Btbd9", "Slc17a7", "Ntf5", "Camk2n1")
)

In [273]:
go_gene <- list()
for (i in 1:length(go_terms)) {
        inner_list <- go_terms[[i]]
        inner_list <- inner_list[inner_list %in% df_merge_filtered$gene]
        go_gene[[i]] <- inner_list
        names(go_gene)[[i]] <- names(go_terms)[[i]]
}

In [274]:
gene_list <- unlist(go_gene)

In [275]:
gene_list <- gene_list[gene_list %in% df_merge$gene]

In [ ]:
counts <- LayerData(seurat_object_whole_clustering, assay = "SCT")

In [277]:
gene_list <- gene_list[gene_list %in% rownames(counts)]

In [278]:
average_expression_profiles_by_cluster <- as.matrix(counts[gene_list, ])

In [279]:
# Crée une colonne d'identifiant unique pour chaque combinaison
seurat_object_whole_clustering@meta.data$group_id <- with(seurat_object_whole_clustering@meta.data, 
                                             paste(condition, time, seurat_custom_clusters, sep = "_"))

In [280]:
# Définir l'ordre souhaité
clusters_order <- c("0.0", "0.1", "0.2", "0.3", "1", "2", "3.0", "3.1", "3.2", "3.3", "4")
conditions_order <- c("CTRL", "SE")
times_order <- c("5", "10", "20", "40")

# Initialiser les vecteurs
clusters_vec <- c()
condition_vec <- c()
time_vec <- c()

for (cond in conditions_order) {
  for (tm in times_order) {
    for (cl in clusters_order) {
      clusters_vec <- c(clusters_vec, cl)
      condition_vec <- c(condition_vec, cond)
      time_vec <- c(time_vec, tm)
    }
  }
}

In [281]:
meta <- seurat_object_whole_clustering@meta.data

In [282]:
cellsPerCluster <- split(rownames(meta), meta[,"group_id"])
Expression_byCellType <- sapply(cellsPerCluster,
                                     function(cells) rowMeans(average_expression_profiles_by_cluster[,cells]))

In [283]:
Expression_byCellType <- Expression_byCellType[,stringr::str_order(colnames(Expression_byCellType), numeric=T)]

In [284]:
# Scale the data to z-scores
mat <- as.matrix(t(scale(t(Expression_byCellType))))

In [ ]:
mat <- MinMax(as.matrix(mat), -2.5, 2.5)
col_fun <- colorRamp2(seq(min(mat), max(mat), length = 50), colorRampPalette(rev(brewer.pal(n = 10, name = "RdYlBu")))(50))
hb <- HeatmapAnnotation(Condition = condition_vec,
                        Time = time_vec,
                        Cluster = clusters_vec,
                        col = list(Cluster = c("0.0" = "#238b45", "1" = "#cccccc", "2" = "#f768a1", "4" = "#bae1ff", "0.2" = "#bae4b3",
                                               "0.3" = "#edf8e9", "0.1" = "#74c476", "3.0" = "#fdf498", "3.1" = "#807dba", "3.2" = "#4a1486", "3.3" = "#ac8fff"),
                                   Condition = c("SE" = "#ff9300", "CTRL" = "#7a81ff"),
                                   Time = c("5" = "#feedde", "10" = "#fdbe85", "20" = "#fd8d3c", "40" = "#d94701")),
                        gp = gpar(fontsize = 20),
                        simple_anno_size = unit(1.5, "cm"))
lgd <- list(title = "Expression levels", legend_height = unit(6, "cm"), grid_width = unit(1, "cm"), labels_gp = gpar(fontsize = 20), title_gp = gpar(fontsize = 18), title_position = "leftcenter-rot")

In [286]:
split_list <- c()
for (i in names(lengths(go_gene))) {
    split_list <- c(split_list, rep(i, lengths(go_gene)[i]))
    }

In [ ]:
options(repr.plot.width=80, repr.plot.height=50) # To set the figure size in Jupyter
pdf('/home/adufour/work/plots/epilepsie/gene_heatmap_pathway_grouped_go_terms_vchristophe_new_list.pdf',width=40,height=80)
hm <- draw(ComplexHeatmap::Heatmap(mat,
                                   col = col_fun,
                                   cluster_columns = FALSE,
                                   cluster_rows = TRUE,
                                   show_row_names = TRUE,
                                   row_names_gp=grid::gpar(fontface = "italic", fontsize=32),
                                   show_column_names = FALSE,
                                   top_annotation = hb,
                                   row_split = split_list,
                                   row_gap = unit(4, "mm"),
                                   column_title = NULL,
                                   heatmap_legend_param = lgd), padding = unit(c(2, 2, 2, 10), "mm"))
dev.off()

pdf 
  2